# OpenAlex 2026-01-16 (Renly snapshot) — data structure survey

Maps `/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet/` — **627 GB**, read-only,
owned by `renly`.

**Nothing here reads a dataset in full.** Parquet footers carry row counts and schemas and cost
~15 ms each; gzip headers cost ~0.05 s. Everything that needs actual values samples a
configurable handful of partitions and says so. Running every cell takes a couple of minutes
and touches well under a gigabyte.

The three things worth knowing before using this snapshot, all established below:

1. **`works_semantic.work_id` is a URL** (`https://openalex.org/W…`) while every other table
   uses the bare accession (`W…`). A join on `work_id` between it and anything else returns
   **zero rows** and raises no error.
2. **`works` names its key `id`; all thirteen other work tables name it `work_id`.**
3. **Partitions are aligned across all fourteen work datasets** — `part_0000` holds the same
   works everywhere — so a part-by-part join needs no shuffle. That is worth a lot on a
   205 GB table, and it is easy not to notice.

In [1]:
# (1) Setup
import os, gzip, time, math
from collections import OrderedDict
import numpy as np, pandas as pd
import pyarrow.parquet as pq

ROOT = "/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet"
OUT  = "/project/jevans/Dawoon/Science of Science"
os.makedirs(OUT, exist_ok=True)

# How many partitions to touch when a question needs real values rather than metadata.
# Every cell that uses this says what it costs. Raise it for a firmer estimate.
N_SAMPLE_PARTS = 8

def human(n):
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or u == "TB":
            return f"{n:,.1f} {u}"
        n /= 1024

def dir_bytes(d):
    """Total size and file count, from stat() only — no reads."""
    tot = n = 0
    for dirpath, _, files in os.walk(d):
        for f in files:
            try:
                tot += os.path.getsize(os.path.join(dirpath, f)); n += 1
            except OSError:
                pass
    return tot, n

assert os.path.isdir(ROOT), f"cannot see {ROOT}"
print(f"root : {ROOT}")
print(f"       readable: {os.access(ROOT, os.R_OK)}   writable: {os.access(ROOT, os.W_OK)}"
      "  (read-only share — write your outputs elsewhere)")
print(f"out  : {OUT}")

root : /project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet
       readable: True   writable: False  (read-only share — write your outputs elsewhere)
out  : /project/jevans/Dawoon/Science of Science


## (2) The top level

Two kinds of thing sit side by side: gzipped CSV entity tables, and one directory of
partitioned parquet. The `_progress_dataset` directories are ingest bookkeeping from the job
that built the snapshot, not data.

In [2]:
# (2) Top-level map  — stat() only, no reads
rows = []
for name in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, name)
    if os.path.isdir(p):
        b, n = dir_bytes(p)
        rows.append(dict(name=name, kind="dir", files=n, bytes=b))
    else:
        rows.append(dict(name=name, kind="file", files=1, bytes=os.path.getsize(p)))
top = pd.DataFrame(rows).sort_values("bytes", ascending=False).reset_index(drop=True)
top["size"] = top.bytes.map(human)
print(f"{len(top)} top-level entries, {human(top.bytes.sum())} total\n")
display(top[["name", "kind", "files", "size"]])

32 top-level entries, 626.3 GB total



,name,kind,files,size
0,works,dir,18676,566.3 GB
1,works_au_affs_fixed.csv.gz,file,1,38.0 GB
2,authors.csv.gz,file,1,5.7 GB
3,authors_x_concepts.csv.gz,file,1,5.1 GB
4,authors_topics.csv.gz,file,1,3.2 GB
5,authors_topic_share.csv.gz,file,1,3.0 GB
6,authors_affiliations.csv.gz,file,1,2.6 GB
7,authors_counts_by_year.csv.gz,file,1,1.5 GB
8,authors_ids.csv.gz,file,1,793.3 MB
9,sources.csv.gz,file,1,93.9 MB


## (3) The fourteen work datasets

`works/` holds fourteen parquet datasets, each split into the same number of `part_NNNN.parquet`
files. Row counts come from the footers of a sample of partitions and are scaled to the full
dataset — footers are exact per file, so the only approximation is which partitions were read.

In [3]:
# (3) Partitioned parquet datasets under works/
#     cost: N_SAMPLE_PARTS footers per dataset (~15 ms each), no column data
WORKS = f"{ROOT}/works"
datasets = sorted(d for d in os.listdir(WORKS) if os.path.isdir(f"{WORKS}/{d}"))

rows = []
schemas = {}
for d in datasets:
    parts = sorted(f for f in os.listdir(f"{WORKS}/{d}") if f.endswith(".parquet"))
    pick = parts[:: max(1, len(parts) // N_SAMPLE_PARTS)][:N_SAMPLE_PARTS]
    n_rows = 0
    for f in pick:
        n_rows += pq.ParquetFile(f"{WORKS}/{d}/{f}").metadata.num_rows
    est = n_rows / len(pick) * len(parts)
    sch = pq.ParquetFile(f"{WORKS}/{d}/{parts[0]}").schema_arrow
    schemas[d] = sch
    b, _ = dir_bytes(f"{WORKS}/{d}")
    rows.append(dict(dataset=d, parts=len(parts), bytes=b,
                     rows_per_part=n_rows / len(pick), est_rows=est, n_cols=len(sch.names)))

works_ds = pd.DataFrame(rows).sort_values("bytes", ascending=False).reset_index(drop=True)
works_ds["size"] = works_ds.bytes.map(human)
works_ds["est_rows_M"] = (works_ds.est_rows / 1e6).round(1)
print(f"{len(works_ds)} datasets, {human(works_ds.bytes.sum())}, "
      f"~{works_ds.est_rows.sum()/1e9:.1f}B rows total "
      f"(from {N_SAMPLE_PARTS} partitions each)\n")
display(works_ds[["dataset", "parts", "size", "n_cols", "est_rows_M"]])

14 datasets, 566.3 GB, ~12.2B rows total (from 8 partitions each)



,dataset,parts,size,n_cols,est_rows_M
0,works,1334,204.1 GB,13,395.7
1,works_semantic,1334,186.6 GB,5,395.7
2,concepts,1334,47.6 GB,3,4136.4
3,referenced_works,1334,27.1 GB,2,2218.5
4,related_works,1334,16.7 GB,2,1701.5
5,work_add_infor,1334,14.9 GB,12,395.7
6,locations,1334,13.8 GB,7,377.9
7,ids,1334,13.2 GB,6,395.7
8,primary_locations,1334,11.4 GB,7,311.1
9,topics,1334,8.8 GB,3,736.0


### Schemas

One row per column of every work dataset. This is the reference table for "which file has the
field I need".

In [4]:
# (4) Every column of every work dataset
rows = []
for d, sch in schemas.items():
    for name, typ in zip(sch.names, sch.types):
        rows.append(dict(dataset=d, column=name, dtype=str(typ)))
cols = pd.DataFrame(rows)
print(f"{len(cols)} columns across {cols.dataset.nunique()} datasets\n")
for d in datasets:
    c = cols[cols.dataset == d]
    print(f"{d}  ({len(c)} cols)")
    print("    " + ", ".join(f"{r.column}:{r.dtype}" for r in c.itertuples()) + "\n")
cols.to_csv(f"{OUT}/openalex_2026_schemas.csv", index=False)
print(f"-> {OUT}/openalex_2026_schemas.csv")

83 columns across 14 datasets

best_oa_locations  (7 cols)
    work_id:string, source_id:string, landing_page_url:string, pdf_url:string, is_oa:bool, version:string, license:string

biblio  (5 cols)
    work_id:string, volume:string, issue:string, first_page:string, last_page:string

concepts  (3 cols)
    work_id:string, concept_id:string, score:double

ids  (6 cols)
    work_id:string, openalex:string, doi:string, mag:string, pmid:string, pmcid:null

locations  (7 cols)
    work_id:string, source_id:string, landing_page_url:string, pdf_url:string, is_oa:bool, version:string, license:string

mesh  (6 cols)
    work_id:string, descriptor_ui:string, descriptor_name:string, qualifier_ui:string, qualifier_name:string, is_major_topic:bool

open_access  (5 cols)
    work_id:string, is_oa:bool, oa_status:string, oa_url:string, any_repository_has_fulltext:bool

primary_locations  (7 cols)
    work_id:string, source_id:string, landing_page_url:string, pdf_url:string, is_oa:bool, version:string

## (5) Join keys — read this before joining anything

Two naming inconsistencies in this snapshot will silently produce empty joins.

In [5]:
# (5) The key column of each dataset, and the FORM its values take
#     cost: one row from each dataset
rows = []
for d in datasets:
    parts = sorted(f for f in os.listdir(f"{WORKS}/{d}") if f.endswith(".parquet"))
    f = pq.ParquetFile(f"{WORKS}/{d}/{parts[0]}")
    names = f.schema_arrow.names
    key = next((c for c in ("work_id", "id") if c in names), names[0])
    v = str(next(f.iter_batches(batch_size=1, columns=[key])).to_pylist()[0][key])
    rows.append(dict(dataset=d, key_column=key, example=v,
                     form="URL" if v.startswith("http") else "accession"))
keys = pd.DataFrame(rows)
display(keys)

odd_name = keys[keys.key_column != keys.key_column.mode()[0]]
odd_form = keys[keys.form != keys.form.mode()[0]]
print("\nTWO TRAPS:")
print(f"  key column   : {', '.join(odd_name.dataset)} uses "
      f"'{odd_name.key_column.iloc[0]}', the other {len(keys)-len(odd_name)} use 'work_id'")
print(f"  value form   : {', '.join(odd_form.dataset)} stores the FULL URL, the other "
      f"{len(keys)-len(odd_form)} store the bare accession")
print("\n  Normalise before joining — this returns 0 rows otherwise, with no error:")
print("      df['work_id'] = df['work_id'].str.rsplit('/', n=1).str[-1]")

,dataset,key_column,example,form
0,best_oa_locations,work_id,W3002427681,accession
1,biblio,work_id,W3002427681,accession
2,concepts,work_id,W3002427681,accession
3,ids,work_id,W3002427681,accession
4,locations,work_id,W3002427681,accession
5,mesh,work_id,W1915717499,accession
6,open_access,work_id,W3002427681,accession
7,primary_locations,work_id,W3002427681,accession
8,referenced_works,work_id,W3002427681,accession
9,related_works,work_id,W3002427681,accession



TWO TRAPS:
  key column   : works uses 'id', the other 13 use 'work_id'
  value form   : works_semantic stores the FULL URL, the other 13 store the bare accession

  Normalise before joining — this returns 0 rows otherwise, with no error:
      df['work_id'] = df['work_id'].str.rsplit('/', n=1).str[-1]


### Are the partitions aligned?

If `part_0000` of every dataset holds the same works, a join can run partition by partition —
no shuffle, no index, and 205 GB never has to be in memory at once. Worth checking rather than
assuming.

In [6]:
# (6) Partition alignment  — reads ONE key column from part_0000 of each dataset
def norm(s):
    return s.astype(str).str.rsplit("/", n=1).str[-1]

base = norm(pq.read_table(f"{WORKS}/works/part_0000.parquet",
                          columns=["id"]).to_pandas()["id"])
base_set = set(base)
print(f"reference: works/works part_0000 — {len(base_set):,} distinct work ids\n")

rows = []
for d in datasets:
    key = keys.loc[keys.dataset == d, "key_column"].iloc[0]
    x = norm(pq.read_table(f"{WORKS}/{d}/part_0000.parquet",
                           columns=[key]).to_pandas()[key])
    u = set(x)
    rows.append(dict(dataset=d, rows=len(x), distinct_ids=len(u),
                     in_reference=len(u & base_set),
                     pct=100 * len(u & base_set) / max(1, len(u)),
                     rows_per_work=len(x) / max(1, len(u))))
align = pd.DataFrame(rows)
align["aligned"] = align.pct > 99.5
display(align.round(2))

if align.aligned.all():
    print("\nAll fourteen partition identically: part_NNNN holds the same works everywhere.")
    print("A part-by-part join needs no shuffle — process one index at a time.")
else:
    print("\nNOT aligned:", ", ".join(align.loc[~align.aligned, "dataset"]))
print("\n'rows_per_work' > 1 marks a one-to-many table (topics, concepts, references, …);")
print("== 1 marks one row per work (works, ids, works_semantic, biblio).")

reference: works/works part_0000 — 400,000 distinct work ids



,dataset,rows,distinct_ids,in_reference,pct,rows_per_work,aligned
0,best_oa_locations,289183,289183,289183,100.0,1.00,True
1,biblio,67826,67826,67826,100.0,1.00,True
2,concepts,3149475,199963,199963,100.0,15.75,True
3,ids,400000,400000,400000,100.0,1.00,True
4,locations,426240,365550,365550,100.0,1.17,True
5,mesh,472442,15933,15933,100.0,29.65,True
6,open_access,400000,400000,400000,100.0,1.00,True
7,primary_locations,364876,364876,364876,100.0,1.00,True
8,referenced_works,2368616,55883,55883,100.0,42.39,True
9,related_works,488792,48706,48706,100.0,10.04,True



All fourteen partition identically: part_NNNN holds the same works everywhere.
A part-by-part join needs no shuffle — process one index at a time.

'rows_per_work' > 1 marks a one-to-many table (topics, concepts, references, …);
== 1 marks one row per work (works, ids, works_semantic, biblio).


## (6) The gzipped entity tables

The non-work entities — authors, institutions, sources, topics, concepts, publishers, funders —
ship as gzipped CSV. Gzip cannot be seeked, so a row count means decompressing the whole file;
`authors.csv.gz` alone is 6 GB compressed. This cell reads only the **header and the first two
rows** of each, which costs milliseconds.

In [7]:
# (7) csv.gz entity tables — header + 2 rows each, nothing more
gz = sorted(f for f in os.listdir(ROOT) if f.endswith(".csv.gz"))
rows = []
for f in gz:
    p = f"{ROOT}/{f}"
    t0 = time.time()
    try:
        with gzip.open(p, "rt", errors="replace") as fh:
            hdr = fh.readline().rstrip("\n")
            first = fh.readline().rstrip("\n")
    except Exception as e:
        rows.append(dict(table=f[:-7], size=human(os.path.getsize(p)),
                         n_cols=np.nan, columns=f"[unreadable: {type(e).__name__}]"))
        continue
    cols_ = hdr.split(",")
    rows.append(dict(table=f[:-7], size=human(os.path.getsize(p)), n_cols=len(cols_),
                     columns=", ".join(cols_)))
ent = pd.DataFrame(rows)
print(f"{len(ent)} gzipped entity tables, "
      f"{human(sum(os.path.getsize(f'{ROOT}/{f}') for f in gz))} compressed\n")
display(ent[["table", "size", "n_cols"]])
print("\ncolumns:")
for r in ent.itertuples():
    print(f"  {r.table:<38} {r.columns[:150]}")
ent.to_csv(f"{OUT}/openalex_2026_entity_tables.csv", index=False)
print(f"\n-> {OUT}/openalex_2026_entity_tables.csv")

29 gzipped entity tables, 60.0 GB compressed



,table,size,n_cols
0,authors,5.7 GB,12
1,authors_affiliations,2.6 GB,5
2,authors_counts_by_year,1.5 GB,5
3,authors_ids,793.3 MB,7
4,authors_topic_share,3.0 GB,6
5,authors_topics,3.2 GB,7
6,authors_x_concepts,5.1 GB,6
7,concepts,4.6 MB,11
8,concepts_ancestors,71.0 B,2
9,concepts_counts_by_year,99.0 B,5



columns:
  authors                                id, orcid, display_name, display_name_alternatives, works_count, cited_by_count, summary_stats, last_known_institution, last_known_institutions, works
  authors_affiliations                   author_id, institution_id, years_json, institution_lineage_json, institution_ror
  authors_counts_by_year                 author_id, year, works_count, cited_by_count, oa_works_count
  authors_ids                            author_id, openalex, orcid, scopus, twitter, wikipedia, mag
  authors_topic_share                    author_id, topic_id, value, subfield_id, field_id, domain_id
  authors_topics                         author_id, topic_id, score, count, subfield_id, field_id, domain_id
  authors_x_concepts                     author_id, concept_id, score, count, level, wikidata
  concepts                               id, wikidata, display_name, level, description, works_count, cited_by_count, image_url, image_thumbnail_url, works_api_url, upd

## (7) What the main tables actually contain

A few rows of the tables most work starts from. `works.abstract_inverted_index` is OpenAlex's
positional encoding; `works_semantic.abstract` is the same text already reconstructed — which
is the one to use.

In [8]:
# (8) Sample rows — one partition, first few rows
def peek(dataset, cols=None, n=3):
    parts = sorted(f for f in os.listdir(f"{WORKS}/{dataset}") if f.endswith(".parquet"))
    f = pq.ParquetFile(f"{WORKS}/{dataset}/{parts[0]}")
    b = next(f.iter_batches(batch_size=n, columns=cols))
    return b.to_pandas()

print("=== works/works ===")
display(peek("works", ["id", "doi", "title", "publication_year", "type",
                       "cited_by_count", "language"]))

print("=== works/works_semantic — note the work_id form ===")
ws = peek("works_semantic", ["work_id", "has_abstract", "abstract"], n=2)
ws["abstract"] = ws["abstract"].astype(str).str[:160] + "…"
display(ws)

print("=== works/ids ===");              display(peek("ids"))
print("=== works/topics ===");           display(peek("topics"))
print("=== works/referenced_works ===");  display(peek("referenced_works"))
print("=== works/work_add_infor ===");    display(peek("work_add_infor"))

=== works/works ===


,id,doi,title,publication_year,type,cited_by_count,language
0,W3002427681,https://doi.org/10.3390/coatings10020109,Montmorillonite-Synergized Water-Based Intumes...,2020,article,34,en
1,W1915717499,https://doi.org/10.1002/jsfa.7056,Fenugreek (<i>Trigonella foenum graecum</i>) s...,2014,article,109,en
2,W3002359967,https://doi.org/10.1016/j.aaf.2020.01.001,The economic contribution of fish and fish tra...,2020,article,158,en


=== works/works_semantic — note the work_id form ===


,work_id,has_abstract,abstract
0,https://openalex.org/W3002427681,True,"<jats:p>In this study, montmorillonite (MMT) w..."
1,https://openalex.org/W1915717499,True,<jats:title>Abstract</jats:title><jats:sec><ja...


=== works/ids ===


...

,work_id,topic_id,score
0,W3002427681,T11360,1.0000
1,W3002427681,T11317,0.9956
2,W3002427681,T10122,0.9535


=== works/referenced_works ===


,work_id,referenced_work_id
0,W3002427681,W2197181935
1,W3002427681,W2912574923
2,W3002427681,W2911254096


=== works/work_add_infor ===


,work_id,corresponding_author_ids_json,corresponding_institution_ids_json,awards_json,funders_json,counts_by_year_json,citation_normalized_percentile_json,fwci,sustainable_development_goals_json,apc_list_json,countries_distinct_count,institutions_distinct_count
0,W3002427681,"[""A5055207901""]","[""I139660479""]",[],[],"[{""year"": 2025, ""cited_by_count"": 6}, {""year"":...","{""value"": 0.86056129, ""is_in_top_1_percent"": f...",2.315116,"[{""id"": ""https://metadata.un.org/sdg/6"", ""disp...","{""value"": 2000, ""currency"": ""CHF"", ""value_usd""...",1,6
1,W1915717499,"[""A5059963899""]","[""I86958956""]","[{""id"": ""https://openalex.org/G4670178431"", ""f...","[{""id"": ""https://openalex.org/F4320321722"", ""d...","[{""year"": 2026, ""cited_by_count"": 1}, {""year"":...","{""value"": 0.90122154, ""is_in_top_1_percent"": f...",3.174767,"[{""id"": ""https://metadata.un.org/sdg/9"", ""disp...","{""value"": 4400, ""currency"": ""USD"", ""value_usd""...",2,6
2,W3002359967,"[""A5006153985""]","[""I135366938""]",[],[],"[{""year"": 2026, ""cited_by_count"": 1}, {""year"":...","{""value"": 0.99603859, ""is_in_top_1_percent"": t...",19.251037,[],None,2,5


## (8) Coverage — years, types, abstracts

The first questions anyone asks of a corpus. Read from a sample of partitions; because
partitioning is by hash rather than by year (established in (6): a single part spans the whole
id range), a sample of partitions is a sample of the corpus.

In [9]:
# (9) Coverage  — reads 3 small columns from N_SAMPLE_PARTS partitions
parts = sorted(f for f in os.listdir(f"{WORKS}/works") if f.endswith(".parquet"))
pick = parts[:: max(1, len(parts) // N_SAMPLE_PARTS)][:N_SAMPLE_PARTS]
frac = len(pick) / len(parts)

d = pd.concat([pq.read_table(f"{WORKS}/works/{f}",
                             columns=["publication_year", "type", "is_retracted"]).to_pandas()
               for f in pick], ignore_index=True)
print(f"{len(d):,} works from {len(pick)}/{len(parts)} partitions "
      f"({frac*100:.1f}% of the corpus)\n")

print("type:")
t = d.type.value_counts()
for k, v in t.head(12).items():
    print(f"  {str(k):<22}{v:>10,}{v/len(d)*100:>7.2f}%   (~{v/frac/1e6:>7.1f}M full corpus)")

print(f"\nretracted: {int(d.is_retracted.sum()):,} ({d.is_retracted.mean()*100:.3f}%)")
print(f"publication_year: {d.publication_year.min():.0f}–{d.publication_year.max():.0f}, "
      f"{d.publication_year.isna().sum():,} null")

yr = d[d.publication_year.between(1900, 2026)].publication_year.value_counts().sort_index()
print("\nworks per decade (scaled to the full corpus):")
for dec, v in (yr.groupby((yr.index // 10) * 10).sum() / frac / 1e6).items():
    if dec >= 1900:
        print(f"  {int(dec)}s   {v:>7.1f}M")

2,373,076 works from 8/1334 partitions (0.6% of the corpus)

type:
  article                1,105,780  46.60%   (~  184.4M full corpus)
  dataset                  470,032  19.81%   (~   78.4M full corpus)
  other                    460,854  19.42%   (~   76.8M full corpus)
  book-chapter             119,763   5.05%   (~   20.0M full corpus)
  dissertation              75,760   3.19%   (~   12.6M full corpus)
  preprint                  37,334   1.57%   (~    6.2M full corpus)
  book                      32,942   1.39%   (~    5.5M full corpus)
  paratext                  16,555   0.70%   (~    2.8M full corpus)
  review                    11,985   0.51%   (~    2.0M full corpus)
  report                    10,620   0.45%   (~    1.8M full corpus)
  libguides                  9,584   0.40%   (~    1.6M full corpus)
  letter                     6,669   0.28%   (~    1.1M full corpus)

retracted: 575 (0.024%)
publication_year: 1010–2050, 135,897 null

works per decade (scaled to the full 

In [10]:
# (10) Abstract coverage — works_semantic.has_abstract, same partition sample
a = pd.concat([pq.read_table(f"{WORKS}/works_semantic/{f}",
                             columns=["has_abstract"]).to_pandas() for f in pick],
              ignore_index=True)
has = a.has_abstract.astype("boolean")
print(f"{len(a):,} works sampled from works_semantic")
print(f"  has_abstract True : {int(has.sum()):,}  ({has.mean()*100:.1f}%)")
print(f"  scaled to corpus  : ~{has.sum()/frac/1e6:.0f}M works with a reconstructed abstract")
print("\nUse works_semantic.abstract — works.abstract_inverted_index is the raw positional")
print("encoding and has to be rebuilt before it is text.")

2,373,076 works sampled from works_semantic
  has_abstract True : 1,435,815  (60.5%)
  scaled to corpus  : ~239M works with a reconstructed abstract

Use works_semantic.abstract — works.abstract_inverted_index is the raw positional
encoding and has to be rebuilt before it is text.


## (9) Summary

Written to the `Science of Science` folder so the next notebook can start from it instead of
re-walking 627 GB.

In [11]:
# (11) Export the survey
summary = (works_ds[["dataset", "parts", "size", "n_cols", "est_rows_M"]]
           .merge(keys[["dataset", "key_column", "form"]], on="dataset")
           .merge(align[["dataset", "rows_per_work", "aligned"]], on="dataset"))
summary.to_csv(f"{OUT}/openalex_2026_works_datasets.csv", index=False)
top.to_csv(f"{OUT}/openalex_2026_top_level.csv", index=False)
print(f"-> {OUT}/openalex_2026_works_datasets.csv")
print(f"-> {OUT}/openalex_2026_top_level.csv")
print(f"-> {OUT}/openalex_2026_schemas.csv")
print(f"-> {OUT}/openalex_2026_entity_tables.csv\n")
display(summary)

print("""
HOW TO READ THIS SNAPSHOT

  works/works              the work table: id, doi, title, year, type, cited_by_count
  works/works_semantic     abstract as TEXT, plus primary_topic_json and keywords_json
  works/ids                doi / mag / pmid / pmcid crosswalk
  works/topics             one row per (work, topic) with a score
  works/referenced_works   the citation edge list
  *.csv.gz                 the non-work entities (authors, institutions, sources, topics)

  Joining:
    - normalise work_id first: .str.rsplit('/', n=1).str[-1]
      works_semantic stores the URL form; everything else stores the accession
    - works calls the key 'id'; the other thirteen call it 'work_id'
    - partitions align, so join part_NNNN to part_NNNN and never shuffle
""")

-> /project/jevans/Dawoon/Science of Science/openalex_2026_works_datasets.csv
-> /project/jevans/Dawoon/Science of Science/openalex_2026_top_level.csv
-> /project/jevans/Dawoon/Science of Science/openalex_2026_schemas.csv
-> /project/jevans/Dawoon/Science of Science/openalex_2026_entity_tables.csv



,dataset,parts,size,n_cols,est_rows_M,key_column,form,rows_per_work,aligned
0,works,1334,204.1 GB,13,395.7,id,accession,1.000000,True
1,works_semantic,1334,186.6 GB,5,395.7,work_id,URL,1.000000,True
2,concepts,1334,47.6 GB,3,4136.4,work_id,accession,15.750289,True
3,referenced_works,1334,27.1 GB,2,2218.5,work_id,accession,42.385269,True
4,related_works,1334,16.7 GB,2,1701.5,work_id,accession,10.035560,True
5,work_add_infor,1334,14.9 GB,12,395.7,work_id,accession,1.000000,True
6,locations,1334,13.8 GB,7,377.9,work_id,accession,1.166024,True
7,ids,1334,13.2 GB,6,395.7,work_id,accession,1.000000,True
8,primary_locations,1334,11.4 GB,7,311.1,work_id,accession,1.000000,True
9,topics,1334,8.8 GB,3,736.0,work_id,accession,2.945511,True



HOW TO READ THIS SNAPSHOT

  works/works              the work table: id, doi, title, year, type, cited_by_count
  works/works_semantic     abstract as TEXT, plus primary_topic_json and keywords_json
  works/ids                doi / mag / pmid / pmcid crosswalk
  works/topics             one row per (work, topic) with a score
  works/referenced_works   the citation edge list
  *.csv.gz                 the non-work entities (authors, institutions, sources, topics)

  Joining:
    - normalise work_id first: .str.rsplit('/', n=1).str[-1]
      works_semantic stores the URL form; everything else stores the accession
    - works calls the key 'id'; the other thirteen call it 'work_id'
    - partitions align, so join part_NNNN to part_NNNN and never shuffle

